# 08 · Raw-file audit and summary
Re-reads every reported simulation value straight from the RASPA output files and checks it against the result tables, then compares your reproduced key results with the original project.

**Runtime: minutes.** **Attach:** the `mofh2_pipeline` dataset and the outputs of notebooks 02–07 (or keep `USE_REFERENCE_UPSTREAM = True` to audit the original runs).

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = False
MAX_HOURS = 11.0
STAGE_QUEUES = []
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

In [ ]:
!python audit_raw.py

In [ ]:
!python compare_reference.py $PKG gcmc_validation.csv h2h2_convention_test.csv topo_gcmc_validation_m2.csv doe_swing_test.csv finetune_130_results.csv finetune_130_real_results.csv train_160_results.csv pi_head_results.csv finetune_77_results.csv exp_gcmc_isotherms.csv

**Headline numbers to check against (original project):** convention-matched CoRE 2024 top 8 within 0.70 g/L; ToBaCCo Spearman 0.98; 160 K term 0.15 g/L (fine-tuned + Widom); 77 K fine-tuning 3.74 → 2.90 g/L on random real MOFs; top 20 all ≥ 50 g/L DOE capacity; MOF-5 DOE swing 52.1 g/L vs 51.9 measured.